In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
import CRUD_Python_Module as CRUD

###########################
# Data Manipulation / Model
###########################
# Initialize Mongo connection
username = "aacuser"
password = "Password"
host = "127.0.0.1"
port = 27017
database = "aac"
collection = "animals"

# Connect to database via CRUD Module
db = CRUD.AnimalShelter(username, password, host, port, database, collection)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))
rescue_types = ['Water Rescue','Mountain or Wilderness Rescue','Disaster or Individual Tracking','Reset']

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    #Title banner
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H2('TREVIN CARLISLE')),
    html.Hr(),
    #Filter buttons
    dcc.RadioItems(
        id='rescue-type-filter',
        options=[{'label':i, 'value':i} for i in rescue_types],
        value='Reset',
        labelStyle={'display':'inline-block'}
    ),
    html.Hr(),
    #Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Set up the features for your interactive data table to make it user-friendly for your client
        editable = False,
        filter_action = "native",
        sort_action = "native",
        sort_mode = "multi",
        column_selectable =  "single",
        row_selectable = "single",
        row_deletable = False,
        selected_columns = [],
        selected_rows = [0],
        page_action = "native",
        page_current = 0,
        page_size = 10
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ]),
    #Logo and signature
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))),
    html.Center(html.H2("This dashboard brought to you by Austin Animal Shelter data, Grazioso Salvare, and Global Rain"))
])

#############################################
# Interaction Between Components / Controller
#############################################

#Button functionality and data table output
@app.callback(Output('datatable-id','data'),
              [Input('rescue-type-filter', 'value')])
def update_dashboard(filter_type):
    if(filter_type == 'Reset'):
        filtered_data = df
    elif(filter_type == 'Water Rescue'):
        filtered_data = df[(df['breed'].str.contains('Labrador Retriever', case=False) |
                            df['breed'].str.contains('Chesa Bay Retr', case=False) | #AAC abbreviates Chesapeake Bay Retriever as Chesa Bay Retr
                            df['breed'].str.contains('Newfoundland', case=False))
                           & (df.sex_upon_outcome == 'Intact Female')
                           & ((df.age_upon_outcome_in_weeks >= 26) & (df.age_upon_outcome_in_weeks <= 156))]
    elif(filter_type == 'Mountain or Wilderness Rescue'):
        filtered_data = df[(df['breed'].str.contains('German Shepherd', case=False) |
                            df['breed'].str.contains('Alaskan Malamute', case=False) |
                            df['breed'].str.contains('Old English Sheepdog', case=False) |
                            df['breed'].str.contains('Siberian Husky', case=False) |
                            df['breed'].str.contains('Rottweiler', case=False))
                           & (df.sex_upon_outcome == 'Intact Male')
                           & ((df.age_upon_outcome_in_weeks >= 26) & (df.age_upon_outcome_in_weeks <= 156))]
    elif(filter_type == 'Disaster or Individual Tracking'):
                filtered_data = df[(df['breed'].str.contains('Doberman Pinsch', case=False) | #AAC abbrevieates Doberman Pinscher as Doberman Pinsch
                            df['breed'].str.contains('German Shepherd', case=False) |
                            df['breed'].str.contains('Golden Retriever', case=False) |
                            df['breed'].str.contains('Bloodhound', case=False) |
                            df['breed'].str.contains('Rottweiler', case=False))
                           & (df.sex_upon_outcome == 'Intact Male')
                           & ((df.age_upon_outcome_in_weeks >= 20) & (df.age_upon_outcome_in_weeks <= 300))]
    
    return filtered_data.to_dict('records')

# Display the breeds of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    graph_df = pd.DataFrame.from_dict(viewData)
    
    return[dcc.Graph(figure=px.pie(graph_df, names='breed', title='Preferred Animals', color_discrete_sequence=px.colors.sequential.RdBu))]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://abrahampelican-dollarrobot-3000.codio.io/proxy/8050/
